# A5: Graph Neural Networks

In this lab, we will implement **Graph Neural Networks (GNNs)** from scratch — without using PyTorch Geometric — to deeply understand the mechanics of message passing.

We will cover the three landmark architectures:

| Model | Year | Key Idea |
|---|---|---|
| **GNN** | 2009 | First general framework: aggregate neighbor features |
| **GCN** | 2017 | Spectral convolution simplified to a symmetric normalized aggregation |
| **GAT** | 2018 | Weighted aggregation using attention scores |

**Application:** We'll apply these models to two tasks:
1. **Genre Prediction** on the MovieLens co-rating graph (node classification)
2. **Recommendation System** (link prediction on a user-item bipartite graph)

---
## Why Graphs?

CNNs work on grids (images). Transformers work on sequences. But many real-world data has **irregular, relational structure**:
- Social networks (users connected to friends)
- Citation networks (papers citing other papers)
- Molecular structures (atoms connected by bonds)
- Recommendation systems (users connected to items they liked)

**The key insight:** A node's label/property is influenced not just by its own features, but by the features of its neighbors.

A **Graph** is defined as `G = (V, E)` where:
- `V` = set of nodes, each with feature vector `h_v`
- `E` = set of edges `(u, v)` encoding relationships
- `A` = adjacency matrix: `A[i][j] = 1` if edge exists between node `i` and `j`

---

## 📚 Papers & Code References

This lab implements all GNN models **from scratch** (no PyTorch Geometric used).

| Model | Paper | Venue | Code Base |
|---|---|---|---|
| **GNN** | Scarselli et al. (2009). *The Graph Neural Network Model* | IEEE Trans. Neural Networks | Foundational concept |
| **GCN** | Kipf & Welling (2017). *Semi-Supervised Classification with Graph Convolutional Networks* | ICLR 2017 | From scratch |
| **GAT** | Veličković et al. (2018). *Graph Attention Networks* | ICLR 2018 | From scratch |
| **MovieLens-100k (node classif.)** | Harper & Konstan (2015). *The MovieLens Datasets* | ACM TIIS | Downloaded from grouplens.org |
| **MovieLens-100k** | Harper & Konstan (2015). *The MovieLens Datasets: History and Context* | ACM TIIS | Downloaded from grouplens.org (open access) |

**External code used:**
- GCN symmetric normalization: Kipf & Welling (2017) Section 2, Eq. 1
- GAT attention: Veličković et al. (2018) Eq. 1–3
- MovieLens loading: built from scratch using pandas

**Paper links:**
- GCN: https://arxiv.org/abs/1609.02907
- GAT: https://arxiv.org/abs/1710.10903
- MovieLens: https://grouplens.org/datasets/movielens/100k/

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.manifold import TSNE
from tqdm import tqdm
import random, os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

set_seed(42)

---
## Part 1: Understanding Graphs

Before building a GNN, let's build intuition by visualizing Zachary's Karate Club — a small, classic social network dataset where members split into two groups after a conflict.

This is the "MNIST of GNNs" — every GNN tutorial starts here!

In [ ]:
# Zachary's Karate Club graph
G_karate = nx.karate_club_graph()

# Visualize
plt.figure(figsize=(10, 7))
labels_karate = nx.get_node_attributes(G_karate, 'club')
colors = ['skyblue' if v == 'Mr. Hi' else 'salmon' for v in labels_karate.values()]
pos = nx.spring_layout(G_karate, seed=42)
nx.draw(G_karate, pos, node_color=colors, with_labels=True,
        node_size=600, font_size=9, edge_color='gray')
plt.title('Karate Club Network\nBlue = Mr. Hi group | Red = Officer group')
plt.axis('off'); plt.show()

print(f'Nodes: {G_karate.number_of_nodes()}')
print(f'Edges: {G_karate.number_of_edges()}')

# Convert to adjacency matrix
A = nx.to_numpy_array(G_karate)
print(f'Adjacency matrix shape: {A.shape}')

plt.figure(figsize=(5, 5))
plt.imshow(A, cmap='Blues')
plt.title('Adjacency Matrix')
plt.colorbar(); plt.show()

---
## Part 2: The MovieLens Co-rating Graph

Instead of Cora's citation network, we build a **movie co-rating graph** from MovieLens-100k:

- **1,682 nodes**: each node is a movie
- **Edges**: two movies are connected if **the same user rated both** (co-rated)
- **Node features**: genre one-hot vector (18 genres) + release year (normalized)
- **Labels**: primary genre (Action, Drama, Comedy, Thriller, ...)

**Why is this like Cora?**

```
Cora                          MovieLens Co-rating Graph
─────────────────────         ────────────────────────────
Paper node                →   Movie node
Cite edge (A cites B)     →   Co-rating edge (same user rated A & B)
Bag-of-words features     →   Genre + year features
7 research topic labels   →   Genre labels (18 classes)
```

The intuition is identical: movies that the **same users watch tend to be the same genre** — just like papers in the same field tend to cite each other. GCN can exploit this structure to predict genre even for movies with few ratings.

**Reference:** Harper & Konstan (2015). *The MovieLens Datasets*. ACM TIIS.
Dataset: https://grouplens.org/datasets/movielens/100k/

---

In [ ]:
import urllib.request, zipfile, os
import pandas as pd
import numpy as np
import torch
import scipy.sparse as sp

os.makedirs('data/movielens', exist_ok=True)

# Download MovieLens-100k (already downloaded if Part 5 ran first)
url = 'https://files.grouplens.org/datasets/movielens/ml-100k.zip'
if not os.path.exists('data/movielens/ml-100k/u.data'):
    print('Downloading MovieLens-100k...')
    urllib.request.urlretrieve(url, 'data/movielens/ml-100k.zip')
    with zipfile.ZipFile('data/movielens/ml-100k.zip') as z:
        z.extractall('data/movielens/')
    print('Done!')

# ── Load ratings and movie metadata ──────────────────────────────────────
ratings = pd.read_csv('data/movielens/ml-100k/u.data',
                      sep='\t', names=['user','item','rating','timestamp'])

# Movie info: id, title, year, [unknown, Action, Adventure, Animation,
# Children's, Comedy, Crime, Documentary, Drama, Fantasy, Film-Noir,
# Horror, Musical, Mystery, Romance, Sci-Fi, Thriller, War, Western]
genre_cols = ['unknown','Action','Adventure','Animation',"Children's",
              'Comedy','Crime','Documentary','Drama','Fantasy',
              'Film-Noir','Horror','Musical','Mystery','Romance',
              'Sci-Fi','Thriller','War','Western']

movies = pd.read_csv('data/movielens/ml-100k/u.item', sep='|',
                     encoding='latin-1', header=None,
                     names=['item','title','release_date','video_date',
                            'imdb_url'] + genre_cols)

# ── Build node features: genre one-hot + release year ────────────────────
movies['year'] = movies['release_date'].str.extract(r'(\d{4})').astype(float)
movies['year'] = movies['year'].fillna(movies['year'].median())
movies['year_norm'] = (movies['year'] - movies['year'].min()) /                       (movies['year'].max() - movies['year'].min())

# Feature: [18 genre binary flags] + [year normalized]
feat_cols = genre_cols + ['year_norm']
movie_features = movies[feat_cols].fillna(0).values.astype(np.float32)

# Re-index movies 0..N_MOVIES-1
movie_ids = sorted(movies.item.unique())
mid2idx   = {m: i for i, m in enumerate(movie_ids)}
N_MOVIES  = len(movie_ids)

print(f'Movies: {N_MOVIES}')
print(f'Feature dim: {movie_features.shape[1]}  (18 genres + 1 year)')

# ── Labels: primary genre (first genre = 1) ───────────────────────────────
# Primary genre = the first genre column that is 1
movies_indexed = movies.set_index('item').loc[movie_ids]
genre_matrix   = movies_indexed[genre_cols].values  # (N_MOVIES, 18)

labels_raw = []
for row in genre_matrix:
    idx = np.argmax(row)  # first genre that is 1
    if row[idx] == 0:
        idx = 0  # unknown
    labels_raw.append(idx)

labels = np.array(labels_raw)
n_classes = len(set(labels))
print(f'Genre classes: {n_classes}')
print(f'Genre distribution:')
for i, g in enumerate(genre_cols):
    count = (labels == i).sum()
    if count > 0:
        print(f'  {g:15}: {count:4d} movies')

# ── Build co-rating graph ─────────────────────────────────────────────────
# Edge between movie A and movie B if >= MIN_COMMON users rated both
MIN_COMMON = 5   # at least 5 shared users to create an edge

# Group ratings by user → list of movies each user rated
user_movies = ratings.groupby('user')['item'].apply(list)

rows, cols = [], []
from itertools import combinations
print('\nBuilding co-rating edges (this takes ~30s)...')
for user, items in user_movies.items():
    # Only use movies we have in our index
    valid = [mid2idx[m] for m in items if m in mid2idx]
    for a, b in combinations(valid, 2):
        rows.append(a); cols.append(b)
        rows.append(b); cols.append(a)

# Count co-ratings per pair and threshold
edge_df = pd.DataFrame({'row': rows, 'col': cols})
edge_counts = edge_df.groupby(['row','col']).size().reset_index(name='count')
strong_edges = edge_counts[edge_counts['count'] >= MIN_COMMON]

print(f'Total edges (co-rating >= {MIN_COMMON} users): {len(strong_edges):,}')

# Build adjacency matrix
A_data = np.ones(len(strong_edges))
A_coo  = sp.coo_matrix((A_data,
                         (strong_edges['row'].values, strong_edges['col'].values)),
                        shape=(N_MOVIES, N_MOVIES))
A_movie = torch.FloatTensor(A_coo.toarray()).to(device)

# ── Convert to tensors ────────────────────────────────────────────────────
X_movie = torch.FloatTensor(movie_features).to(device)
Y_movie = torch.LongTensor(labels).to(device)

# Train/val/test split: 20 per class for train
train_mask_m = torch.zeros(N_MOVIES, dtype=torch.bool)
val_mask_m   = torch.zeros(N_MOVIES, dtype=torch.bool)
test_mask_m  = torch.zeros(N_MOVIES, dtype=torch.bool)

for c in range(n_classes):
    idx = (Y_movie == c).nonzero(as_tuple=True)[0]
    if len(idx) >= 20:
        train_mask_m[idx[:20]] = True
    elif len(idx) > 0:
        train_mask_m[idx[:len(idx)//2]] = True

remaining = (~train_mask_m).nonzero(as_tuple=True)[0]
n_val = min(200, len(remaining)//2)
val_mask_m[remaining[:n_val]] = True
test_mask_m[remaining[n_val:n_val+500]] = True

train_mask_m = train_mask_m.to(device)
val_mask_m   = val_mask_m.to(device)
test_mask_m  = test_mask_m.to(device)

print(f'\nSplit — Train: {train_mask_m.sum().item()} | Val: {val_mask_m.sum().item()} | Test: {test_mask_m.sum().item()}')

# ── Visualize the co-rating graph (sample) ───────────────────────────────
import networkx as nx
import matplotlib.pyplot as plt

# Sample 80 movies for visualization
sample_idx = np.random.choice(N_MOVIES, 80, replace=False)
A_sample   = A_coo.toarray()[np.ix_(sample_idx, sample_idx)]
G_sample   = nx.from_numpy_array(A_sample)

genre_colors = plt.cm.tab20(np.linspace(0, 1, n_classes))
node_colors  = [genre_colors[labels[i]] for i in sample_idx]
node_labels_map = {i: genre_cols[labels[sample_idx[i]]][:3] for i in range(len(sample_idx))}

plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G_sample, seed=42, k=0.4)
nx.draw(G_sample, pos, node_color=node_colors, node_size=200,
        with_labels=False, edge_color='gray', alpha=0.7, width=0.5)

# Legend
handles = [plt.scatter([],[], c=[genre_colors[i]], s=80,
           label=genre_cols[i]) for i in range(n_classes)
           if (labels[sample_idx] == i).any()]
plt.legend(handles=handles, loc='upper right', fontsize=7, ncol=2)
plt.title('MovieLens Co-rating Graph (80 movies sample)\n'
          'Connected = rated by ≥5 same users | Color = primary genre', fontsize=12)
plt.axis('off'); plt.tight_layout(); plt.show()


---
## Part 3: GCN — Genre Prediction on MovieLens Co-rating Graph

### Message Passing Intuition

The key idea of all GNNs is **message passing**: update each node's representation by aggregating information from its neighbors.

```
Iteration 1:
  Node v gathers features from all 1-hop neighbors → h_v^(1)

Iteration 2:
  Node v gathers h^(1) from neighbors → h_v^(2)
  Now h_v^(2) contains info from 2-hop neighborhood!
```

### GCN Formula (Kipf & Welling, 2017)

GCN simplifies spectral graph convolution to:

$$H^{(l+1)} = \sigma\left( \tilde{D}^{-1/2} \tilde{A} \tilde{D}^{-1/2} H^{(l)} W^{(l)} \right)$$

Where:
- $\tilde{A} = A + I$ — adjacency matrix **with self-loops** (a node also reads its own features!)
- $\tilde{D}$ — degree matrix of $\tilde{A}$
- $\tilde{D}^{-1/2} \tilde{A} \tilde{D}^{-1/2}$ — **symmetric normalization** to prevent nodes with many neighbors from dominating
- $W^{(l)}$ — learnable weight matrix

**Why self-loops?** Without self-loops, a node forgets its own features after each layer — it only aggregates neighbor info.

In [ ]:
def normalize_adjacency(A):
    """
    Compute the symmetrically normalized adjacency: D^{-1/2} (A+I) D^{-1/2}
    This is the 'GCN trick' that makes training stable.
    """
    # Add self-loops
    A_tilde = A + torch.eye(A.size(0), device=A.device)
    # Degree matrix
    D = A_tilde.sum(dim=1)
    D_inv_sqrt = torch.diag(D.pow(-0.5))
    # Symmetric normalization
    return D_inv_sqrt @ A_tilde @ D_inv_sqrt


class GCNLayer(nn.Module):
    """A single GCN layer: H_new = σ(Â H W)"""
    def __init__(self, in_features, out_features):
        super().__init__()
        self.W = nn.Linear(in_features, out_features, bias=False)
        nn.init.xavier_uniform_(self.W.weight)

    def forward(self, H, A_norm):
        # Step 1: Linear transform
        HW = self.W(H)             # (N, out_features)
        # Step 2: Aggregate from neighbors (matrix multiply with normalized A)
        out = A_norm @ HW          # (N, out_features)
        return out


class GCN(nn.Module):
    """2-layer GCN for node classification."""
    def __init__(self, in_features, hidden_dim, n_classes, dropout=0.5):
        super().__init__()
        self.layer1   = GCNLayer(in_features, hidden_dim)
        self.layer2   = GCNLayer(hidden_dim, n_classes)
        self.dropout  = nn.Dropout(dropout)

    def forward(self, X, A_norm):
        # Layer 1: feature extraction
        h = F.relu(self.layer1(X, A_norm))
        h = self.dropout(h)
        # Layer 2: classification
        out = self.layer2(h, A_norm)
        return out, h  # return logits and hidden embeddings

In [ ]:
# Precompute normalized adjacency
A_norm = normalize_adjacency(A_movie)

# Train GCN
gcn = GCN(in_features=X_movie.shape[1], hidden_dim=64, n_classes=n_classes).to(device)
optimizer_gcn = torch.optim.Adam(gcn.parameters(), lr=0.01, weight_decay=5e-4)

train_mask = train_mask_m.to(device)
val_mask   = val_mask_m.to(device)
test_mask  = test_mask_m.to(device)

gcn_train_acc, gcn_val_acc = [], []

for epoch in range(200):
    gcn.train()
    logits, _ = gcn(X_movie, A_norm)
    loss = F.cross_entropy(logits[train_mask], Y_movie[train_mask])
    optimizer_gcn.zero_grad(); loss.backward(); optimizer_gcn.step()

    gcn.eval()
    with torch.no_grad():
        logits, _ = gcn(X_movie, A_norm)
        train_acc = (logits[train_mask].argmax(1) == Y_movie[train_mask]).float().mean().item()
        val_acc   = (logits[val_mask].argmax(1) == Y_movie[val_mask]).float().mean().item()
    gcn_train_acc.append(train_acc)
    gcn_val_acc.append(val_acc)
    if (epoch+1) % 50 == 0:
        print(f'Epoch {epoch+1:3d} | Loss: {loss:.4f} | Train: {train_acc:.4f} | Val: {val_acc:.4f}')

# Test accuracy
gcn.eval()
with torch.no_grad():
    logits, gcn_embeddings = gcn(X_movie, A_norm)
    test_acc = (logits[test_mask].argmax(1) == Y_movie[test_mask]).float().mean().item()
print(f'\n✅ GCN Genre Prediction Test Accuracy: {test_acc*100:.2f}%')

---
## Part 4: GAT — Graph Attention Network

### The Problem with GCN

GCN uses **fixed, symmetric weights** for aggregation. A node with 100 neighbors and a node with 2 neighbors are treated symmetrically — but not all neighbors are equally important!

**GAT** (Veličković et al., 2018) solves this by computing **attention scores** between nodes:

$$\alpha_{ij} = \text{softmax}_j \left( \text{LeakyReLU}\left( \mathbf{a}^T [W h_i \| W h_j] \right) \right)$$

$$h_i^{\prime} = \sigma\left( \sum_{j \in \mathcal{N}(i)} \alpha_{ij} W h_j \right)$$

Where:
- `||` denotes concatenation
- `a` is a learnable attention vector
- `α_ij` is the normalized attention weight: how much node `i` should attend to node `j`

**Multi-head attention**: just like Transformers, GAT uses K parallel attention heads and concatenates (or averages) their outputs.

In [ ]:
class GATLayer(nn.Module):
    """Single-head GAT layer with attention mechanism."""
    def __init__(self, in_features, out_features, dropout=0.6, alpha=0.2):
        super().__init__()
        self.W  = nn.Linear(in_features, out_features, bias=False)
        # Attention vector: takes concatenated [Wh_i || Wh_j] → scalar
        self.a  = nn.Linear(2 * out_features, 1, bias=False)
        self.leaky_relu = nn.LeakyReLU(alpha)
        self.dropout    = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.W.weight)
        nn.init.xavier_uniform_(self.a.weight)

    def forward(self, H, A):
        N   = H.size(0)
        Wh  = self.W(H)             # (N, out_features)

        # Compute attention coefficients
        # For each pair (i, j): concat [Wh_i || Wh_j] and apply attention vector
        Wh_i = Wh.unsqueeze(1).expand(-1, N, -1)   # (N, N, F)
        Wh_j = Wh.unsqueeze(0).expand(N, -1, -1)   # (N, N, F)
        e    = self.leaky_relu(self.a(torch.cat([Wh_i, Wh_j], dim=-1)).squeeze(-1))  # (N, N)

        # Mask out non-edges (set to -inf so softmax → 0)
        mask = (A == 0) & (~torch.eye(N, dtype=torch.bool, device=A.device))
        e    = e.masked_fill(mask, float('-inf'))

        # Normalize with softmax
        alpha = F.softmax(e, dim=1)          # (N, N)
        alpha = self.dropout(alpha)

        # Weighted aggregation
        out = alpha @ Wh                     # (N, out_features)
        return out, alpha


class GAT(nn.Module):
    """2-layer GAT with multi-head attention."""
    def __init__(self, in_features, hidden_dim, n_classes, n_heads=8, dropout=0.6):
        super().__init__()
        self.heads = nn.ModuleList([
            GATLayer(in_features, hidden_dim, dropout) for _ in range(n_heads)
        ])
        self.out_layer = GATLayer(hidden_dim * n_heads, n_classes, dropout, alpha=0.2)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, X, A):
        X = self.dropout(X)
        # Layer 1: multi-head, concatenate outputs
        head_outs = [F.elu(head(X, A)[0]) for head in self.heads]
        h = torch.cat(head_outs, dim=-1)    # (N, hidden_dim * n_heads)
        h = self.dropout(h)
        # Layer 2: single head output
        out, attn = self.out_layer(h, A)
        return out, h, attn

In [ ]:
# Note: For large graphs, full N×N attention is expensive.
# MovieLens (1682 nodes) is manageable on GPU but will be slow on CPU.

gat = GAT(in_features=X_movie.shape[1], hidden_dim=8, n_classes=n_classes, n_heads=8).to(device)
optimizer_gat = torch.optim.Adam(gat.parameters(), lr=5e-3, weight_decay=5e-4)

gat_val_acc = []

for epoch in range(200):
    gat.train()
    logits, _, _ = gat(X_movie, A_movie)
    loss = F.cross_entropy(logits[train_mask], Y[train_mask])
    optimizer_gat.zero_grad(); loss.backward(); optimizer_gat.step()

    gat.eval()
    with torch.no_grad():
        logits, gat_embeddings, attn = gat(X_movie, A_movie)
        val_acc = (logits[val_mask].argmax(1) == Y_movie[val_mask]).float().mean().item()
    gat_val_acc.append(val_acc)
    if (epoch+1) % 50 == 0:
        print(f'Epoch {epoch+1:3d} | Loss: {loss:.4f} | Val Acc: {val_acc:.4f}')

gat.eval()
with torch.no_grad():
    logits, _, _ = gat(X_movie, A_movie)
    test_acc_gat = (logits[test_mask].argmax(1) == Y_movie[test_mask]).float().mean().item()
print(f'\n✅ GAT Genre Prediction Test Accuracy: {test_acc_gat*100:.2f}%')

## Visualize: GCN vs GAT Embeddings (t-SNE)

In [ ]:
from sklearn.manifold import TSNE

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors_map = plt.cm.tab20(np.linspace(0, 1, n_classes))

for ax, (name, emb) in zip(axes, [
    ('GCN', gcn_embeddings),
    ('GAT', gat_embeddings)
]):
    emb_np     = emb.cpu().detach().numpy()
    labels_np  = Y_movie.cpu().numpy()
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    proj = tsne.fit_transform(emb_np)
    for c in range(n_classes):
        mask = labels_np == c
        if mask.any():
            ax.scatter(proj[mask, 0], proj[mask, 1],
                       c=[colors_map[c]], label=genre_cols[c], alpha=0.7, s=12)
    ax.set_title(f'{name} Movie Embeddings (t-SNE)', fontsize=13)
    ax.legend(fontsize=6, markerscale=2, ncol=2)
    ax.axis('off')

plt.suptitle('Learned Movie Representations — Genre Clustering\n'
             '(No genre labels used during GNN training)', fontsize=14)
plt.tight_layout(); plt.show()


---
## Part 5: Application — Recommendation System (MovieLens-100k)

### Graph-based Recommendation

Recommendation systems are one of the most impactful real-world applications of GNNs. The key insight:

> A **user-item interaction** is naturally a **bipartite graph**: users on one side, movies on the other, edges represent ratings ≥ 4 stars (liked).

```
User 1 ──── Toy Story
    └──────── Fargo
User 2 ──── Fargo
    └──────── Pulp Fiction
→ GNN: User1 & User2 both liked Fargo → similar taste → User2 might like Toy Story
```

### Dataset: MovieLens-100k (real data)

**MovieLens-100k** (Harper & Konstan, 2015) is a widely-used benchmark in recommender systems research:
- **943 users**, **1,682 movies**, **100,000 ratings** (1–5 stars)
- Collected by GroupLens Research at the University of Minnesota
- We treat ratings ≥ 4 as **positive interactions** (user liked the movie)

**Reference:** Harper, F. M., & Konstan, J. A. (2015). *The MovieLens Datasets: History and Context*. ACM TIIS, 5(4).
Dataset: https://grouplens.org/datasets/movielens/100k/

---

In [ ]:
import urllib.request, zipfile, os
import pandas as pd

os.makedirs('data/movielens', exist_ok=True)

# Download MovieLens-100k
url = 'https://files.grouplens.org/datasets/movielens/ml-100k.zip'
if not os.path.exists('data/movielens/ml-100k/u.data'):
    print('Downloading MovieLens-100k (~5 MB)...')
    urllib.request.urlretrieve(url, 'data/movielens/ml-100k.zip')
    with zipfile.ZipFile('data/movielens/ml-100k.zip') as z:
        z.extractall('data/movielens/')
    print('Done!')

# Load ratings: user_id, item_id, rating, timestamp
ratings = pd.read_csv('data/movielens/ml-100k/u.data',
                      sep='\t', names=['user','item','rating','timestamp'])
# Load movie titles
movies = pd.read_csv('data/movielens/ml-100k/u.item', sep='|', encoding='latin-1',
                     usecols=[0,1], names=['item','title'])

print(f'Users:   {ratings.user.nunique()}')
print(f'Movies:  {ratings.item.nunique()}')
print(f'Ratings: {len(ratings):,}')
print(f'\nSample ratings:')
print(ratings.head())
print(f'\nSample movies:')
print(movies.head())

# ── Build bipartite graph ─────────────────────────────────────────────────
# Keep only positive interactions (rating >= 4 = liked)
pos_ratings = ratings[ratings.rating >= 4].copy()
print(f'\nPositive interactions (rating ≥ 4): {len(pos_ratings):,}')

# Re-index users 0..N_USERS-1, items 0..N_ITEMS-1
user_ids  = sorted(pos_ratings.user.unique())
item_ids  = sorted(pos_ratings.item.unique())
user2idx  = {u: i for i, u in enumerate(user_ids)}
item2idx  = {it: i for i, it in enumerate(item_ids)}

N_USERS = len(user_ids)
N_ITEMS = len(item_ids)
N_NODES = N_USERS + N_ITEMS
FEAT_DIM = 32

print(f'\nGraph nodes: {N_USERS} users + {N_ITEMS} movies = {N_NODES} total')

# Map to global node IDs: users 0..N_USERS-1, movies N_USERS..N_NODES-1
pos_ratings = pos_ratings.copy()
pos_ratings['u_idx'] = pos_ratings.user.map(user2idx)
pos_ratings['i_idx'] = pos_ratings.item.map(item2idx) + N_USERS  # offset by N_USERS

# All positive edges as (user_node, movie_node)
all_pos = list(zip(pos_ratings.u_idx, pos_ratings.i_idx))
random.shuffle(all_pos)

# Train / test split: 80 / 20
split      = int(0.8 * len(all_pos))
train_pos  = all_pos[:split]
test_pos   = all_pos[split:]
pos_set    = set(all_pos)

print(f'Train edges: {len(train_pos):,} | Test edges: {len(test_pos):,}')

def sample_negatives(n, pos_set, n_users, n_items):
    """Sample (user, item) pairs with NO positive interaction."""    negs = []
    while len(negs) < n:
        u = random.randint(0, n_users - 1)
        i = random.randint(0, n_items - 1) + n_users
        if (u, i) not in pos_set:
            negs.append((u, i))
    return negs

train_neg = sample_negatives(len(train_pos), pos_set, N_USERS, N_ITEMS)
test_neg  = sample_negatives(len(test_pos),  pos_set, N_USERS, N_ITEMS)

# ── Build adjacency matrix (train edges only) ─────────────────────────────
A_rec = torch.zeros(N_NODES, N_NODES)
for u, v in train_pos:
    A_rec[u][v] = 1; A_rec[v][u] = 1  # symmetric

A_rec_norm = normalize_adjacency(A_rec.to(device))

# Learnable node embeddings (users + movies)
user_emb = nn.Embedding(N_USERS, FEAT_DIM)
item_emb = nn.Embedding(N_ITEMS, FEAT_DIM)
node_features = torch.cat([user_emb.weight, item_emb.weight], dim=0).detach().to(device)

# Refresh node_features after each embedding update in training
print(f'\nNode feature shape: {node_features.shape}')
print(f'  (each user + movie has a learnable {FEAT_DIM}-dim embedding)')

# Show some movie titles for context
print(f'\nSample movies in the graph:')
for idx in list(item2idx.keys())[:5]:
    title = movies[movies.item == idx].title.values[0] if len(movies[movies.item == idx]) else '?'
    print(f'  item {idx} → node {item2idx[idx]+N_USERS}: {title}')


In [ ]:
class RecGCN(nn.Module):
    """GCN encoder for MovieLens recommendation.
    Input: learnable user/item embeddings + graph structure
    Output: enriched node embeddings for link prediction
    """
    def __init__(self, in_features=32, hidden=64, out=32):
        super().__init__()
        self.layer1 = GCNLayer(in_features, hidden)
        self.layer2 = GCNLayer(hidden, out)

    def forward(self, X, A_norm):
        h = F.relu(self.layer1(X, A_norm))
        return self.layer2(h, A_norm)

    def predict(self, embeddings, u, v):
        """Dot product → sigmoid score for edge (u, v)."""""
        return torch.sigmoid((embeddings[u] * embeddings[v]).sum(dim=-1))


# ── Build model with learnable embeddings ────────────────────────────────
user_emb_layer = nn.Embedding(N_USERS, FEAT_DIM).to(device)
item_emb_layer = nn.Embedding(N_ITEMS, FEAT_DIM).to(device)

rec_model  = RecGCN(in_features=FEAT_DIM, hidden=64, out=32).to(device)
opt_rec    = torch.optim.Adam(
    list(rec_model.parameters()) +
    list(user_emb_layer.parameters()) +
    list(item_emb_layer.parameters()),
    lr=1e-2
)

for epoch in range(30):
    rec_model.train()
    # Refresh node features from learnable embeddings
    node_feat = torch.cat([user_emb_layer.weight,
                           item_emb_layer.weight], dim=0)
    emb = rec_model(node_feat, A_rec_norm)

    pos_u = torch.tensor([e[0] for e in train_pos], device=device)
    pos_v = torch.tensor([e[1] for e in train_pos], device=device)
    neg_u = torch.tensor([e[0] for e in train_neg], device=device)
    neg_v = torch.tensor([e[1] for e in train_neg], device=device)

    pos_scores = rec_model.predict(emb, pos_u, pos_v)
    neg_scores = rec_model.predict(emb, neg_u, neg_v)
    loss = (-torch.log(pos_scores + 1e-8).mean()
            - torch.log(1 - neg_scores + 1e-8).mean())

    opt_rec.zero_grad(); loss.backward(); opt_rec.step()

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d} | Loss: {loss:.4f}')

# ── Evaluate: AUC + Recall@10 ────────────────────────────────────────────
rec_model.eval()
with torch.no_grad():
    node_feat = torch.cat([user_emb_layer.weight,
                           item_emb_layer.weight], dim=0)
    emb = rec_model(node_feat, A_rec_norm)

    tpu = torch.tensor([e[0] for e in test_pos], device=device)
    tpv = torch.tensor([e[1] for e in test_pos], device=device)
    tnu = torch.tensor([e[0] for e in test_neg], device=device)
    tnv = torch.tensor([e[1] for e in test_neg], device=device)
    pos_sc = rec_model.predict(emb, tpu, tpv)
    neg_sc = rec_model.predict(emb, tnu, tnv)

auc = ((pos_sc.unsqueeze(1) > neg_sc.unsqueeze(0)).float().mean()).item()
print(f'\n✅ Recommendation AUC: {auc:.4f}')
print(f'   Random baseline AUC: 0.5000')

# ── Recall@10: for a sample of users ─────────────────────────────────────
K = 10
user_test = {}
for u, v in test_pos:
    user_test.setdefault(u, []).append(v)

recalls = []
with torch.no_grad():
    for user_node, true_items in list(user_test.items())[:100]:  # sample 100 users
        item_nodes = torch.arange(N_USERS, N_NODES, device=device)
        u_emb_rep  = emb[user_node].unsqueeze(0).expand(N_ITEMS, -1)
        scores     = torch.sigmoid((emb[item_nodes] * u_emb_rep).sum(dim=-1))
        top_k      = scores.topk(K).indices.cpu().numpy() + N_USERS
        n_hits     = sum(1 for it in true_items if it in top_k)
        recalls.append(n_hits / len(true_items))

recall_at_10 = np.mean(recalls)
print(f'   Recall@10:           {recall_at_10:.4f}')
print(f'   Random Recall@10:    {K/N_ITEMS:.4f}')

# ── Show top-10 recommendations for a sample user ─────────────────────────
sample_user = list(user_test.keys())[0]
with torch.no_grad():
    item_nodes = torch.arange(N_USERS, N_NODES, device=device)
    u_emb_rep  = emb[sample_user].unsqueeze(0).expand(N_ITEMS, -1)
    scores     = torch.sigmoid((emb[item_nodes] * u_emb_rep).sum(dim=-1))
    top10_idx  = scores.topk(10).indices.cpu().numpy()

print(f'\n🎬 Top-10 recommendations for User {user_ids[sample_user]}:')
for rank, idx in enumerate(top10_idx):
    orig_item = item_ids[idx]
    title = movies[movies.item == orig_item].title.values
    title = title[0] if len(title) else '?'
    liked = '✓ (in test set)' if (sample_user, idx + N_USERS) in set(test_pos) else ''
    print(f'  {rank+1:2d}. {title[:50]:50} {liked}')


# Exercises

## Exercise 1: Number of Layers — Depth vs Over-smoothing

Each GCN layer aggregates 1-hop neighbors. With `k` layers, you aggregate `k`-hop neighborhoods. More layers = more context — but too many layers cause **over-smoothing** (all nodes become identical).

a) Train GCN with 1, 2, 3, 4, and 5 layers on MovieLens. Record test accuracy for each.

| # Layers | Test Accuracy |
|---|---|
| 1 | ? |
| 2 | ? |
| 3 | ? |
| 4 | ? |
| 5 | ? |

b) After training each model, compute the **pairwise cosine similarity** between movie embeddings. A high average similarity indicates over-smoothing. Plot similarity vs. # layers.

c) At what depth does over-smoothing start to hurt accuracy? Explain why this happens mechanically.

---

## Exercise 2: GCN vs GAT — When does Attention Help?

a) Compare GCN and GAT on the full MovieLens test set:

| Model | Test Accuracy | Training Time (200 epochs) |
|---|---|---|
| GCN | ? | ? |
| GAT (1 head) | ? | ? |
| GAT (8 heads) | ? | ? |

b) Visualize the attention weights α_ij for a few sample nodes. Which neighbors does the model attend to most? Do they share the same genre label?

```python
# Get attention weights for node 0
attn_0 = attn[0].cpu().detach().numpy()
top_k = np.argsort(attn_0)[-5:]  # top-5 attended neighbors
```

c) Explain in your own words: in what type of graph would you expect GAT to significantly outperform GCN?

---

## Exercise 3: MLP Baseline — Do We Even Need the Graph?

A common sanity check: train an MLP on **only node features** (ignoring edges) and compare.

a) Implement and train a 2-layer MLP:
```python
class MLP(nn.Module):
    def __init__(self, in_features, hidden, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(hidden, n_classes)
        )
    def forward(self, X):
        return self.net(X)
```

b) Compare test accuracy: MLP vs GCN vs GAT.

| Model | Test Accuracy |
|---|---|
| MLP (no graph) | ? |
| GCN | ? |
| GAT | ? |

c) By how much does the graph structure improve accuracy over just node features? What does this tell you about the value of relational information?

---

## Exercise 4 (Challenge): LightGCN — When Less is More

**LightGCN** (He et al., SIGIR 2020) removes the weight matrix `W` and non-linear activation from GCN — making it **better** for recommendation.

**GCN layer:** `H' = σ( D^{-1/2} A D^{-1/2} · H · W )` — has W and σ

**LightGCN layer:** `H' = D^{-1/2} A D^{-1/2} · H` — pure propagation

Final embedding = weighted average across all layers: `e_final = (1/K+1) Σ H^(k)`

a) Implement `LightGCNLayer` and `LightGCN` (see lab code) and train on MovieLens-100k with `embed_dim=32`, `n_layers=3`, Adam `lr=1e-2`, 30 epochs.

b) Fill in the comparison table:

| Model | # Trainable Params | AUC | Recall@10 | Time/epoch |
|---|---|---|---|---|
| RecGCN (with W) | ? | ? | ? | ? |
| LightGCN (no W) | ? | ? | ? | ? |

c) Train LightGCN with `n_layers = {1, 2, 3, 4}` and plot Recall@10 vs number of layers. Does over-smoothing appear? Why or why not?

d) LightGCN has no weight matrix W — what are the only trainable parameters left? Write 2–3 sentences explaining what LightGCN is actually learning to optimize.

---

## Submission

Submit your work to GitHub. Your repository should contain:

### 1. Training Script (`train.py`)

```bash
# Train GCN on MovieLens
python3 train.py --model gcn      --dataset movielens --epochs 200 --train

# Train GAT on MovieLens
python3 train.py --model gat      --dataset movielens --epochs 200 --heads 8 --train

# Train MLP baseline (no graph)
python3 train.py --model mlp      --dataset movielens --epochs 200 --train

# Train RecGCN for recommendation
python3 train.py --model recgcn   --dataset movielens --epochs 30  --train

# Train LightGCN for recommendation (Exercise 4)
python3 train.py --model lightgcn --dataset movielens --epochs 30  --n-layers 3 --train

# Layer ablation (Exercise 1)
python3 train.py --model gcn      --dataset movielens --epochs 200 --n-layers 1 --train
python3 train.py --model gcn      --dataset movielens --epochs 200 --n-layers 5 --train
```

### 2. `README.md`

Your `README.md` must include:

**Commands used** (exact commands you ran)

**Results table:**

| Model | Test Accuracy | # Params | Training Time | Notes |
|---|---|---|---|---|
| MLP (no graph) | ? | ? | ? | no-graph baseline |
| GCN (2 layers) | ? | ? | ? | |
| GAT (8 heads) | ? | ? | ? | |
| RecGCN | ? | ? | ? | recommendation |
| LightGCN | ? | ? | ? | no W, no σ |

**Visualizations** (include in README or as separate image files):
- t-SNE embeddings: GCN vs GAT
- Over-smoothing plot: test accuracy + cosine similarity vs # layers
- Attention weight visualization (Exercise 2b)
- LightGCN layer ablation: Recall@10 vs n_layers

**Discussion** (3–5 sentences): When would you use a GNN instead of a simple MLP? Give a concrete example from a domain outside of movies (e.g., biology, social networks, traffic routing).